In [15]:
import pandas as pd
import numpy as np
import math

In [16]:
def pred_acd(x):
    amplitude = 33.52533001
    angular_frequency = 4.79692269
    phase_offset = 2.31996204
    vertical_offset = 105.43875151
    return amplitude * np.sin(angular_frequency * x + phase_offset) + vertical_offset


In [17]:
def pred_ace(x):
    slope = 1.07686462
    vertical_offset = 97.16703124
    return slope * x + vertical_offset
    

In [18]:
def pred_bcd(x):
    vertical_offset = 1.40993318e+02
    amplitude = 3.26487772e+01
    angular_frequency = 4.79431594e+00
    phase_offset = 2.33387954e+00
    slope_magnitude = 6.11884394e+02
    horizontal_shift = 2.83217492e-02
    period = 9.97641125e-02
    return vertical_offset + amplitude * np.sin(angular_frequency * x + phase_offset) - slope_magnitude * ((x - horizontal_shift) % period)

In [19]:
def pred_bce(x):
    vertical_offset = 1.32146719e+02
    slope_magnitude = 5.92088606e+02
    horizontal_shift = 1.26169775e-01
    period = 1.00159422e-01
    return vertical_offset - slope_magnitude * ((x - horizontal_shift) % period)

In [20]:
df = pd.read_json("traffic.jsonl", lines=True)
df["depature"] = pd.to_datetime(df["depature"], format="%H:%M")
df["arrival"] = pd.to_datetime(df["arrival"], format="%H:%M")
min_depature = 420
max_depature  = 1019
range_depature = max_depature -min_depature
df["depature_minutes"] = (
    df["depature"].dt.hour * 60
    + df["depature"].dt.minute
    )
df["duration"] = (
    (df["arrival"] - df["depature"]).dt.total_seconds() / 60
    )

df["scaled_depature"] = (df["depature_minutes"] - min_depature) / range_depature


In [21]:
df.head()

,road,depature,arrival,depature_minutes,duration,scaled_depature
0,B->C->E,1900-01-01 13:17:00,1900-01-01 15:25:00,797,128.0,0.629382
1,A->C->E,1900-01-01 07:07:00,1900-01-01 08:47:00,427,100.0,0.011686
2,A->C->E,1900-01-01 07:59:00,1900-01-01 09:32:00,479,93.0,0.098497
3,B->C->E,1900-01-01 14:21:00,1900-01-01 16:29:00,861,128.0,0.736227
4,B->C->D,1900-01-01 10:09:00,1900-01-01 11:13:00,609,64.0,0.315526


In [22]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1031 entries, 0 to 1030
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   road              1031 non-null   str           
 1   depature          1031 non-null   datetime64[us]
 2   arrival           1031 non-null   datetime64[us]
 3   depature_minutes  1031 non-null   int32         
 4   duration          1031 non-null   float64       
 5   scaled_depature   1031 non-null   float64       
dtypes: datetime64[us](2), float64(2), int32(1), str(1)
memory usage: 44.4 KB


In [23]:
df_errors = pd.DataFrame({
    "route": pd.Series(dtype="string"),
    "depature": pd.Series(dtype="datetime64[ns]"),
    "error": pd.Series(dtype="float64")
    })
df_errors.info()


<class 'pandas.DataFrame'>
RangeIndex: 0 entries
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype         
---  ------    --------------  -----         
 0   route     0 non-null      string        
 1   depature  0 non-null      datetime64[ns]
 2   error     0 non-null      float64       
dtypes: datetime64[ns](1), float64(1), string(1)
memory usage: 132.0 bytes


In [24]:

for  index, row in df.iterrows():
    df_errors.loc[index, "route"] = row["road"]
    df_errors.loc[index, "depature"] = row.depature
    if row.road == "A->C->E":
        y_hat = pred_ace(row.scaled_depature)
        y = row.duration

    elif row.road == "A->C->D":
        y_hat = pred_acd(row.scaled_depature)
        y = row.duration
    elif row.road == "B->C->E":
        y_hat = pred_bce(row.scaled_depature)
        y = row.duration
    elif row.road == "B->C->D":
        y_hat = pred_bcd(row.scaled_depature)
        y = row.duration

    df_errors.loc[index, "error"] = abs(y - y_hat)




In [25]:

df_errors.head()

,route,depature,error
0,B->C->E,1900-01-01 13:17:00,2.716577
1,A->C->E,1900-01-01 07:07:00,2.820384
2,A->C->E,1900-01-01 07:59:00,4.273100
3,B->C->E,1900-01-01 14:21:00,1.241724
4,B->C->D,1900-01-01 10:09:00,2.187851


Min avgangstid: 420 min, maks avgangstid: 1019 min

In [26]:
df_errors.to_csv("./errors.csv",)

In [27]:
import pandas as pd

input_time = pd.to_datetime(
    input("Enter a time in HH:MM format: "),
    format="%H:%M"
)


In [29]:

import numpy as np 
# Bruk samme dato som input_time, slik at bare klokkeslettet avgjør
prev_errors = pd.read_csv(
    "./errors.csv",
    index_col=0,
    parse_dates=["depature"]
).rename(columns={"depature": "time"})

neighbors_acd = prev_errors.loc[
    (prev_errors["route"] == "A->C->D")
    & (prev_errors["time"] >= input_time - pd.Timedelta(minutes=15))
    & (prev_errors["time"] <= input_time + pd.Timedelta(minutes=15))
]


neighbors_ace = prev_errors.loc[
    (prev_errors["route"] == "A->C->E")
    & (prev_errors["time"] >= input_time - pd.Timedelta(minutes=15))
    & (prev_errors["time"] <= input_time + pd.Timedelta(minutes=15))
]

neighbors_bcd = prev_errors.loc[
    (prev_errors["route"] == "B->C->D")
    & (prev_errors["time"] >= input_time - pd.Timedelta(minutes=15))
    & (prev_errors["time"] <= input_time + pd.Timedelta(minutes=15))
]

neighbors_bce = prev_errors.loc[
    (prev_errors["route"] == "B->C->E")
    & (prev_errors["time"] >= input_time - pd.Timedelta(minutes=15))
    & (prev_errors["time"] <= input_time + pd.Timedelta(minutes=15))
]

neighbors_acd_errors = np.array(neighbors_acd["error"]) 
neighbors_ace_errors = np.array(neighbors_ace["error"]) 
neighbors_bcd_errors = np.array(neighbors_bcd["error"]) 
neighbors_bce_errors = np.array(neighbors_bce["error"]) 

acd_lower, acd_upper = np.percentile(neighbors_acd_errors, [10, 90])
ace_lower, ace_upper = np.percentile(neighbors_ace_errors, [10, 90])
bcd_lower, bcd_upper = np.percentile(neighbors_bcd_errors, [10, 90])
bce_lower, bce_upper = np.percentile(neighbors_bce_errors, [10, 90])

scaled_input_time = (input_time.hour * 60 + input_time.minute - min_depature) / range_depature
predicted_acd = pred_acd(scaled_input_time)
predicted_ace = pred_ace(scaled_input_time)
predicted_bcd = pred_bcd(scaled_input_time)
predicted_bce = pred_bce(scaled_input_time)

print(f"A->C->D: {predicted_acd:.2f} min - {acd_lower:.2f} + {acd_upper:.2f})")
print(f"A->C->E: {predicted_ace:.2f} min - {ace_lower:.2f} + {ace_upper:.2f})")
print(f"B->C->D: {predicted_bcd:.2f} min - {bcd_lower:.2f} + {bcd_upper:.2f})")
print(f"B->C->E: {predicted_bce:.2f} min - {bce_lower:.2f} + {bce_upper:.2f})")


A->C->D: 76.87 min - 0.69 + 6.82)
A->C->E: 97.58 min - 1.77 + 7.36)
B->C->D: 78.46 min - 0.74 + 5.58)
B->C->E: 98.11 min - 0.86 + 3.55)
